# Notebook 04 — Surface Occupation Energy (SOE) Analysis

Computes SOEs relative to pristine CuNi and explores correlations with
electronic descriptors (Bader charge, electronegativity difference).

**Key outputs:**
- Correlation heatmap of SOEs
- Scatter plots: SOE vs. dopant Bader charge
- Radar (spider) charts for pathway selectivity

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src import (
    SYSTEMS_LABEL_MAP,
    SYSTEM_COLORS,
    compute_soe,
    normalize_zscore,
    compute_pearson_correlation,
    build_feature_dataframe,
    plot_soe_scatter_vs_feature,
    plot_correlation_heatmap,
    plot_spider_chart,
)

## Electronic descriptor data

In [ ]:
# Bader charges on dopant atom (e) and electronegativity differences
dopant_charges = {
    'CuNi': -0.02, 'Pd@Ni': -0.3224, 'Pd@Cu': -0.3327,
    'Fe@Ni': 0.3732, 'Ag@Cu': -0.1198, 'Ag@Ni': -0.1100,
    'Pt@Ni': -0.5869, 'Au@Cu': -0.4458, 'Au@Ni': -0.4377,
}

electroneg_diffs = {
    'CuNi': 0.0, 'Pd@Ni': 0.2945, 'Pd@Cu': 0.2945,
    'Fe@Ni': -0.0755, 'Ag@Cu': 0.0245, 'Ag@Ni': 0.0245,
    'Pt@Ni': 0.3745, 'Au@Cu': 0.6345, 'Au@Ni': 0.6345,
}

## Load SOE data and compute correlations

In [ ]:
df_soe = pd.read_csv('../data/df_dopant_SOEs_cunisite2.csv')

mol_cols = ['0_N2','2_N','3_N2H','4_NHNH','5_NNH2','6_NNH3',
            '7_NH2NH','8_N2H4','9_NH2','10_NH','11_NH3']

df_soe = build_feature_dataframe(
    df_soe, systems=list(df_soe['system']),
    dopant_charges=dopant_charges,
    electroneg_diffs=electroneg_diffs,
)
df_soe.head()

In [ ]:
fig = plot_correlation_heatmap(
    df_soe[mol_cols],
    title='SOE Pearson Correlation Matrix',
    figsize=(18, 16),
)
plt.show()

## SOE vs. dopant Bader charge

In [ ]:
systems_display = [SYSTEMS_LABEL_MAP.get(s, s) for s in df_soe['system']]

for mol in ['0_N2', '11_NH3', '3_N2H']:
    fig = plot_soe_scatter_vs_feature(
        feature_vals=list(df_soe['dopant_charge']),
        soe_vals=list(df_soe[mol]),
        systems=systems_display,
        xlabel='Dopant Bader Charge (e)',
        ylabel=f'{mol} SOE (eV)',
        title=f'{mol} SOE vs. Dopant Charge',
    )
    plt.show()

## Radar chart: normalised SOEs by pathway

In [ ]:
assoc_alt_cols = ['0_N2','3_N2H','4_NHNH','7_NH2NH','8_N2H4','9_NH2','11_NH3']

# Exclude CuNi reference (SOE = 0)
df_plot = df_soe[df_soe['system'] != '0_CuNi'].copy()
df_plot['group'] = [SYSTEMS_LABEL_MAP.get(s, s) for s in df_plot['system']]

df_norm = normalize_zscore(df_plot[assoc_alt_cols])
df_norm['group'] = df_plot['group'].values
df_norm = df_norm[['group'] + assoc_alt_cols]

fig = plot_spider_chart(df_norm, group_col='group', figsize=(14, 14))
plt.suptitle('Normalised SOEs — Associative Alternating Pathway', y=1.02, fontsize=14)
plt.show()